# 07 — Network statistics for Task A (issues #11, #12, #13)

Notebooks 04–05 built three patient-similarity graphs for **Long Length of Stay** (Task A). This notebook describes them: how they are shaped, what groups they form, and whether they connect patients who share an outcome.

| Graph | Built from | Role |
|---|---|---|
| **safe** | Each patient's diagnoses *before* the prediction time | The honest graph we'd actually use |
| **leaky** | Each patient's diagnoses over their *whole* record, including the future | A deliberate cheat, used as a control |
| **null** | The safe graph with its edges randomly rewired. Every node keeps its exact degree, and the edge weights are shuffled | A "structure means nothing" control |

The notebook answers three questions, one per issue:

1. **#11 — Shape.** How many connections do patients have, how clustered is the graph, and which patients sit in the middle?
2. **#12 — Groups.** Do Louvain and Leiden find the same groups of patients? Do those groups differ in their long-stay rate?
3. **#13 — Alignment with the outcome.** Are connected patients more likely to share the same outcome than chance predicts (*homophily*)? And do connected patients have similar CLMBR fingerprints (*Dirichlet energy*)?

**One null graph is not enough.** The null graph is random, so any single copy could land high or low by luck. Every null number below is the mean over **10 differently seeded null graphs**, with a range covering the middle 95% of them. A real graph only counts as "different from random" if it falls outside that range.

**Data rule.** This runs locally against the on-device dataset. Only per-graph summary numbers (`results/aggregate/network_stats_task_a.csv`) and figures built from them (`results/aggregate/figures/07_*.png`) are committed. Per-patient tables such as centrality scores and community memberships stay in memory and are never written out.

In [ ]:
# Standard bootstrap cell (identical in every notebook -- see notebooks/README.md):
# find the repo root, put it on the import path, and point EHRSHOT_DATA_ROOT
# at the local, on-device copy of the data.
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

In [ ]:
import random
import time

import igraph as ig
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from sklearn.metrics import normalized_mutual_info_score, roc_auc_score

from ehrgraph.data.loader import open_database
from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.eval.baseline import load_clmbr_features
from ehrgraph.graphs.null_model import build_null_graph
from ehrgraph.graphs.similarity import build_similarity_graph, graph_summary
from ehrgraph.graphs.vectorize import build_node_vectors
from ehrgraph.netstats.centrality import centrality_suite, degree_distribution
from ehrgraph.netstats.communities import compare_partitions
from ehrgraph.netstats.homophily import (
    class_insensitive_homophily,
    dirichlet_energy,
    edge_homophily,
)
from ehrgraph.netstats.structure import structural_stats

TASK = "guo_los"
# The primary graph configuration (PROJECT_GUIDE.md Part 4 items 6 and 12):
# binary presence per patient, diagnoses only, TF-IDF + cosine, mutual k-NN, k=10.
GRAPH_VARIANT = "cond_binary_perpatient_tfidf_mutual_k10"
N_NULL = 10  # null-graph seeds 0..9; seed 0 is "the" null used for per-node figures

results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# One fixed color per graph, used in every figure. These are the first three
# slots of a colorblind-validated palette.
COLORS = {"safe": "#2a78d6", "leaky": "#eb6834", "null": "#1baf7a"}
INK, INK_2, INK_3, GRID = "#0b0b0b", "#52514e", "#8a8983", "#e6e5e0"
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.edgecolor": INK_3,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": INK_3,
        "ytick.color": INK_3,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "legend.frameon": False,
        "legend.labelcolor": INK_2,
        "font.size": 10,
    }
)


def save(fig, name, dpi=150):
    # Figures are summaries only: no patient IDs, no labels on individual nodes.
    fig.savefig(figures_dir / name, dpi=dpi, bbox_inches="tight")

## Building the three graphs

These are the same steps as notebook 05, now called through `build_similarity_graph`. The graph-building code lives in exactly one tested place, so this notebook, notebook 05, and the later Task B repeat can't quietly drift apart.

In [ ]:
t0 = time.time()
labels = pd.read_csv(
    assets / "benchmark" / TASK / "labeled_patients.csv", parse_dates=["prediction_time"]
)
db = open_database(str(meds_reader_path))
labels = labels[labels["patient_id"].isin(set(db))].reset_index(drop=True)
nodes = list(zip(labels["patient_id"], labels["prediction_time"], strict=True))
node_label = dict(zip(nodes, labels["value"].astype(int), strict=True))
assert len(node_label) == len(nodes), "duplicate (patient_id, prediction_time) nodes"

safe_matrix, safe_nodes, _ = build_node_vectors(db, nodes, table="condition", censor=True)
leaky_matrix, leaky_nodes, _ = build_node_vectors(db, nodes, table="condition", censor=False)

graphs = {
    "safe": build_similarity_graph(safe_matrix, safe_nodes),
    "leaky": build_similarity_graph(leaky_matrix, leaky_nodes),
}
nulls = [build_null_graph(graphs["safe"], seed=s) for s in range(N_NULL)]
graphs["null"] = nulls[0]

prevalence = np.mean(list(node_label.values()))
print(f"{len(nodes):,} nodes, long-stay prevalence {prevalence:.3f}")
print(f"built 2 real graphs + {N_NULL} null graphs in {time.time() - t0:.0f}s")
pd.DataFrame({name: graph_summary(g) for name, g in graphs.items()}).T

**Aligning CLMBR fingerprints to the graph.** Dirichlet energy (Part 3) needs one CLMBR vector per node, in the same order as the graph's nodes. The fingerprints are matched on `(patient_id, prediction_time)`, the same key the baseline check in notebook 01 validated. The match must cover every node, and the notebook stops if it doesn't.

In [ ]:
features = load_clmbr_features(assets / "features" / "clmbr_features.pkl")
features = features.set_index(["patient_id", "prediction_time"])["embedding"]
missing = [n for n in nodes if n not in features.index]
assert not missing, f"{len(missing)} nodes have no CLMBR embedding"
X = np.stack([features.loc[n] for n in nodes]).astype(np.float64)
# Center each dimension: an offset shared by every patient isn't "similarity"
# and would otherwise lower every graph's energy by the same amount.
X = X - X.mean(axis=0)
print("CLMBR feature matrix:", X.shape)

---
# Part 1 — Shape of the graphs (issue #11)

### Whole-graph statistics

Some plain-language definitions:

- **Density**: the share of all possible patient pairs that are actually connected.
- **Clustering** and **transitivity**: when two patients share a neighbor, how often they are also connected to each other (triadic closure, Lecture 3).
- **Degree assortativity**: whether well-connected patients tend to link to other well-connected patients.
- **Path length** and **diameter**: how many hops it takes to get between patients in the largest connected piece.

The null graph keeps every patient's degree, so density and degree numbers are the same as the safe graph *by construction*. The null's clustering and path lengths are the informative part: they show how much structure exists beyond "who has how many links."

In [ ]:
rows = []  # every aggregate number this notebook produces, in the long-format schema


def add(condition, metric, value, ci_low=None, ci_high=None, seed=None):
    rows.append(
        {
            "task": TASK,
            # "null" is on pandas' default missing-value list -- written as null_model
            "condition": "null_model" if condition == "null" else condition,
            "graph_variant": GRAPH_VARIANT,
            "metric": metric,
            "value": float(value),
            "ci_low": ci_low,
            "ci_high": ci_high,
            "seed": seed,
        }
    )


def add_null(metric, values):
    # Null metrics are reported as the mean over N_NULL seeds with an empirical 95% interval.
    values = np.asarray(values, dtype=float)
    lo, hi = np.percentile(values, [2.5, 97.5])
    add("null", metric, values.mean(), lo, hi)
    return values.mean(), lo, hi


t0 = time.time()
struct = {name: structural_stats(graphs[name]) for name in ("safe", "leaky")}
null_struct = [structural_stats(g) for g in nulls]
for name in ("safe", "leaky"):
    for metric, value in (struct[name] | graph_summary(graphs[name])).items():
        add(name, metric, value)
null_summaries = [graph_summary(g) for g in nulls]
struct_table = pd.DataFrame(struct).T.astype(object)  # holds numbers and "lo – hi" text
for metric in null_struct[0]:
    mean, lo, hi = add_null(metric, [s[metric] for s in null_struct])
    struct_table.loc["null (mean of 10)", metric] = mean
    struct_table.loc["null 95% range", metric] = f"{lo:.4g} – {hi:.4g}"
for metric in null_summaries[0]:
    add_null(metric, [s[metric] for s in null_summaries])
print(f"structural stats in {time.time() - t0:.0f}s")
struct_table.T

### Degree distribution

In most real networks, degree follows a long tail: a few hubs have hundreds of connections. **That can't happen here, by design.** Each patient proposes only their 10 most similar patients, and mutual k-NN keeps a link only if both patients chose each other, so **no patient can have more than 10 links**. The histogram below is the whole story, and the interesting part is the left side. A patient with 0 or 1 links is someone nobody else counts among their closest matches: an unusual history, or a very sparse one.

The null graph has exactly the safe graph's degrees, so it isn't drawn separately.

In [ ]:
fig, ax_hist = plt.subplots(figsize=(9, 4))
width = 0.4
for offset, name in ((-width / 2, "safe"), (width / 2, "leaky")):
    counts = degree_distribution(graphs[name]).value_counts().reindex(range(11), fill_value=0)
    ax_hist.bar(
        counts.index + offset,
        counts / counts.sum(),
        width=width - 0.04,
        color=COLORS[name],
        label=name,
    )
ax_hist.set_xlabel("degree (number of links; mutual k-NN caps it at 10)")
ax_hist.set_ylabel("share of all patients")
ax_hist.set_title("Degree distribution (0 = isolated; null = safe by construction)")
ax_hist.set_xticks(range(11))
ax_hist.grid(axis="x", visible=False)
ax_hist.legend()
fig.tight_layout()
save(fig, "07_degree_distribution.png")
plt.show()

### Centrality: which patients sit in the middle?

These are four different notions of "central," with the correction PROJECT_GUIDE.md Module B requires. Edge weights are *similarities*, so betweenness and closeness use `distance = 1 − similarity` instead of the raw weight. Eigenvector centrality uses the similarity directly as a strength.

- Betweenness and closeness here are **exact**. They are computed with igraph's C implementation, about 13 s per graph, where networkx took roughly 22 min for the same numbers. That speed means there's no need to settle for the 500-node sample the guide allowed.
- Closeness and eigenvector centrality are computed inside the largest connected piece only. Patients outside it get "not applicable" rather than an invented value. Eigenvector centrality run over a disconnected graph piles all its weight onto one piece and gives nearly everyone else about 0, which is meaningless.
- Degree is left out of the plot below because the histogram above already shows it.

Per-patient centrality stays in memory. Only its distribution (the curves below) and summary numbers leave this notebook.

In [ ]:
t0 = time.time()
centrality = {name: centrality_suite(graphs[name]) for name in ("safe", "leaky", "null")}
print(f"exact centrality for 3 graphs in {time.time() - t0:.0f}s")

MEASURES = ["degree", "betweenness", "closeness", "eigenvector"]
PLOTTED = ["betweenness", "closeness", "eigenvector"]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
for ax, measure in zip(axes, PLOTTED, strict=True):
    for name in ("safe", "leaky", "null"):
        values = np.sort(centrality[name][measure].dropna().to_numpy())
        ax.plot(
            values,
            np.linspace(0, 1, len(values)),
            color=COLORS[name],
            lw=2,
            ls="--" if name == "null" else "-",
            label=name,
        )
    ax.set_title(measure)
    ax.set_xlabel(f"{measure} centrality")
    if measure in ("betweenness", "eigenvector"):
        ax.set_xscale("symlog", linthresh=1e-4)
axes[0].set_ylabel("share of patients ≤ value")
axes[0].legend(loc="lower right")
fig.suptitle("Centrality distributions (cumulative)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "07_centrality_distributions.png")
plt.show()

summary_rows = []
for name in ("safe", "leaky", "null"):
    c = centrality[name]
    for measure in MEASURES:
        col = c[measure].dropna()
        for stat, value in (
            ("mean", col.mean()),
            ("median", col.median()),
            ("p90", col.quantile(0.9)),
            ("max", col.max()),
        ):
            add(name, f"{measure}_centrality_{stat}", value, seed=0 if name == "null" else None)
            summary_rows.append((name, measure, stat, value))
    # Spearman rank agreement between measures: do they pick the same "central" patients?
    rho = c[MEASURES].corr(method="spearman")
    for i, a in enumerate(MEASURES):
        for b in MEASURES[i + 1 :]:
            add(name, f"spearman_{a}_vs_{b}", rho.loc[a, b], seed=0 if name == "null" else None)
pd.DataFrame(summary_rows, columns=["graph", "measure", "stat", "value"]).pivot_table(
    index=["measure", "stat"], columns="graph", values="value"
)

### Does being central say anything about the outcome?

This is a direct check with one number per graph and measure. Treat centrality alone as a "prediction" of long stay and compute its AUROC:

- **0.5** means centrality carries no information about the outcome.
- **Above 0.5** means central patients are more likely to have a long stay.
- **Below 0.5** means they are less likely.

This is descriptive only. It uses every node's label, so it is **not** a model and none of it feeds training.

In [ ]:
y_all = pd.Series(node_label)
auroc_rows = []
for name in ("safe", "leaky", "null"):
    c = centrality[name]
    for measure in MEASURES:
        col = c[measure].dropna()
        auc = roc_auc_score(y_all.loc[col.index], col)
        add(name, f"{measure}_centrality_auroc_vs_label", auc, seed=0 if name == "null" else None)
        auroc_rows.append((measure, name, auc))
pd.DataFrame(auroc_rows, columns=["measure", "graph", "auroc"]).pivot(
    index="measure", columns="graph", values="auroc"
).round(3)

---
# Part 2 — Groups of patients (issue #12)

**Louvain** and **Leiden** both look for groups of patients that are densely connected inside and sparsely connected to each other, by maximizing *modularity*. Leiden is the newer algorithm and fixes a known Louvain flaw (Louvain can return groups that aren't internally connected).

- **Agreement between them** (adjusted Rand index, ARI): 1 means identical groupings, 0 means no better than chance. High agreement means the groups are a real feature of the graph rather than one algorithm's quirk.
- **Modularity** runs from about 0 (no group structure) up toward 1 (very clean groups). The null gives the baseline modularity that any graph with these degrees reaches by chance.

In [ ]:
t0 = time.time()
comm = {name: compare_partitions(graphs[name], seed=0) for name in ("safe", "leaky")}
null_comm = [compare_partitions(g, seed=0) for g in nulls]
comm["null"] = null_comm[0]
print(f"community detection in {time.time() - t0:.0f}s")

COMM_METRICS = [
    "adjusted_rand_index",
    "n_louvain_communities",
    "n_leiden_communities",
    "louvain_modularity",
    "leiden_modularity",
]
comm_table = pd.DataFrame(
    {n: {m: comm[n][m] for m in COMM_METRICS} for n in ("safe", "leaky")}
).astype(object)
for name in ("safe", "leaky"):
    for m in COMM_METRICS:
        add(name, m, comm[name][m])
for m in COMM_METRICS:
    mean, lo, hi = add_null(m, [r[m] for r in null_comm])
    comm_table.loc[m, "null (mean of 10)"] = mean
    comm_table.loc[m, "null 95% range"] = f"{lo:.3g} – {hi:.3g}"
comm_table

### Do the groups differ in their long-stay rate?

If the graph groups patients in a clinically meaningful way, some Leiden groups should have far more long stays than the overall rate (about 25%) and others far fewer. On a graph whose groups ignore the outcome, every group sits near the overall rate.

The plot shows each group's long-stay rate, sorted, for groups of **at least 20 patients**. Smaller groups are left out: their rates are too noisy to read, and a tiny group's rate says too much about the few patients in it (the README's small-cell rule). Two summary numbers go to the results file:

- **η² (eta squared)**: the share of the variation in outcome that group membership explains. 0 means none.
- **NMI**: normalized mutual information between group and outcome.

In [ ]:
MIN_COMMUNITY = 20


def community_outcome(partition):
    df = pd.DataFrame({"community": partition, "y": y_all.loc[partition.index]})
    sizes = df["community"].map(df["community"].value_counts())
    big = df[sizes >= MIN_COMMUNITY]
    rates = big.groupby("community")["y"].agg(["mean", "size"])
    grand = big["y"].mean()
    between = (rates["size"] * (rates["mean"] - grand) ** 2).sum()
    total = ((big["y"] - grand) ** 2).sum()
    return {
        "rates": rates,
        "eta_squared": between / total,
        "nmi": normalized_mutual_info_score(big["y"], big["community"]),
        "share_in_reported_communities": len(big) / len(df),
    }


outcome = {name: community_outcome(comm[name]["leiden"]) for name in ("safe", "leaky", "null")}
null_outcome = [community_outcome(r["leiden"]) for r in null_comm]
for name in ("safe", "leaky"):
    for m in ("eta_squared", "nmi", "share_in_reported_communities"):
        add(name, f"leiden_outcome_{m}", outcome[name][m])
for m in ("eta_squared", "nmi", "share_in_reported_communities"):
    add_null(f"leiden_outcome_{m}", [o[m] for o in null_outcome])

fig, ax = plt.subplots(figsize=(11, 4.4))
for name in ("safe", "leaky", "null"):
    rates = outcome[name]["rates"]["mean"].sort_values().to_numpy()
    x = np.linspace(0, 1, len(rates))
    ax.plot(
        x,
        rates,
        marker="o",
        ms=4,
        lw=1.5,
        color=COLORS[name],
        ls="--" if name == "null" else "-",
        label=f"{name}: {len(rates)} groups, η² = {outcome[name]['eta_squared']:.3f}",
    )
ax.axhline(prevalence, color=INK_3, lw=1, ls=":")
ax.text(0.0, prevalence + 0.02, f"overall rate {prevalence:.2f}", color=INK_3, fontsize=9)
ax.set_xlabel("Leiden groups, sorted by long-stay rate (position scaled 0–1)")
ax.set_ylabel("long-stay rate within group")
ax.set_ylim(0, 1)
ax.set_title(f"Group long-stay rates (Leiden groups ≥ {MIN_COMMUNITY} patients)")
ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "07_community_outcome_rates.png")
plt.show()

### What the graphs look like

Here is the largest connected piece of each graph, drawn with a force-directed layout: connected patients pull together and unconnected ones push apart. Each dot is a patient, colored only by outcome:

- **violet**: long stay
- **light gray**: not a long stay

No patient is labeled or identifiable. Homophily shows up visually as violet dots clumping together. The null panel shows what "no clumping" looks like.

In [ ]:
def layout_lcc(graph, seed=0):
    lcc = graph.subgraph(max(nx.connected_components(graph), key=len))
    order = list(lcc.nodes)
    index = {n: i for i, n in enumerate(order)}
    ig_graph = ig.Graph(n=len(order), edges=[(index[u], index[v]) for u, v in lcc.edges()])
    coords = np.array(ig_graph.layout_fruchterman_reingold(niter=1500, seed=None).coords)
    return lcc, order, coords


random.seed(0)  # igraph's layout draws from Python's RNG once pointed at it
ig.set_random_number_generator(random)
POS, NEG = "#4a3aa7", "#d4d3cc"
fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
for ax, name in zip(axes, ("safe", "leaky", "null"), strict=True):
    lcc, order, coords = layout_lcc(graphs[name])
    index = {n: i for i, n in enumerate(order)}
    segs = np.array([[coords[index[u]], coords[index[v]]] for u, v in lcc.edges()])
    ax.add_collection(LineCollection(segs, colors=INK_3, linewidths=0.1, alpha=0.08))
    y = np.array([node_label[n] for n in order])
    ax.scatter(*coords[y == 0].T, s=4, c=NEG, linewidths=0)
    ax.scatter(*coords[y == 1].T, s=6, c=POS, linewidths=0)
    ax.set_title(f"{name}: largest piece, {len(order):,} patients")
    ax.set_aspect("equal")
    ax.axis("off")
handles = [
    plt.Line2D([], [], marker="o", ls="", color=POS, label="long stay"),
    plt.Line2D([], [], marker="o", ls="", color=NEG, label="not long stay"),
]
fig.legend(handles=handles, loc="lower center", ncol=2)
fig.tight_layout(rect=(0, 0.05, 1, 1))
save(fig, "07_graph_layouts.png", dpi=80)  # ~7k dots; keeps the file under 500 KB
plt.show()

---
# Part 3 — Do the graphs line up with the outcome? (issue #13)

**Homophily** measures whether connected patients share the same outcome more often than chance.

- **Edge homophily** is the simple version: the share of edges joining two same-outcome patients. It is biased. With 75% of patients *not* having a long stay, most edges connect two non-long-stay patients just by base rate.
- **Class-insensitive homophily** (Lim et al., 2021) removes that bias. For each outcome it takes the rate at which that outcome's patients link to same-outcome patients, then subtracts what random wiring would give. A randomly wired graph scores about **0** at any prevalence, and a perfectly segregated graph scores **1**. This correction matters here because Task B (Hypertension) has a different prevalence (0.14), and the project's central claim compares homophily across both tasks.

**Dirichlet energy** asks the same question about CLMBR fingerprints instead of the label. It measures how much a patient's fingerprint differs from their neighbors', normalized so graphs with different edge counts can be compared. **Lower means connected patients already have similar fingerprints.** That is the regime where averaging over neighbors (the project's graph models) should help least with new information but most with noise.

Both are computed on every node's label. That's fine for a description of the graph, but these same numbers must never be used to choose a model.

In [ ]:
nodelist = nodes  # X's row order -- dirichlet_energy asserts the graph matches it

h = {}
for name in ("safe", "leaky"):
    g = graphs[name]
    ci = class_insensitive_homophily(g, node_label)
    h[name] = {
        "edge_homophily": edge_homophily(g, node_label),
        "class_insensitive_homophily": ci["class_insensitive_homophily"],
        "same_outcome_rate_long_stay": ci["per_class"][1],
        "same_outcome_rate_not_long_stay": ci["per_class"][0],
        "dirichlet_energy_clmbr": dirichlet_energy(g, nodelist, X),
    }
    for metric, value in h[name].items():
        add(name, metric, value)

null_h = []
for g in nulls:
    ci = class_insensitive_homophily(g, node_label)
    null_h.append(
        {
            "edge_homophily": edge_homophily(g, node_label),
            "class_insensitive_homophily": ci["class_insensitive_homophily"],
            "same_outcome_rate_long_stay": ci["per_class"][1],
            "same_outcome_rate_not_long_stay": ci["per_class"][0],
            "dirichlet_energy_clmbr": dirichlet_energy(g, nodelist, X),
        }
    )
h_table = pd.DataFrame(h).astype(object)
for metric in null_h[0]:
    mean, lo, hi = add_null(metric, [r[metric] for r in null_h])
    h_table.loc[metric, "null (mean of 10)"] = mean
    h_table.loc[metric, "null 95% range"] = f"{lo:.4f} – {hi:.4f}"
add("reference", "long_stay_prevalence", prevalence)
h_table

In [ ]:
fig, (ax_h, ax_e) = plt.subplots(1, 2, figsize=(12, 3.8))
panels = (
    (ax_h, "class_insensitive_homophily", "Class-insensitive homophily\n(0 = random)"),
    (ax_e, "dirichlet_energy_clmbr", "Dirichlet energy, CLMBR fingerprints\n(lower = smoother)"),
)
for ax, metric, title in panels:
    null_vals = np.array([r[metric] for r in null_h])
    ypos = {"safe": 2, "leaky": 1, "null": 0}
    ax.hlines(
        ypos["null"], null_vals.min(), null_vals.max(), color=COLORS["null"], lw=6, alpha=0.35
    )
    ax.plot(null_vals, np.zeros_like(null_vals), "o", color=COLORS["null"], ms=5, alpha=0.7)
    for name in ("safe", "leaky"):
        ax.plot(h[name][metric], ypos[name], "o", color=COLORS[name], ms=10)
        ax.annotate(
            f"{h[name][metric]:.3f}",
            (h[name][metric], ypos[name]),
            xytext=(8, 0),
            textcoords="offset points",
            va="center",
            color=INK_2,
        )
    ax.annotate(
        f"{null_vals.mean():.3f} (10 seeds)",
        (null_vals.max(), 0),
        xytext=(8, 0),
        textcoords="offset points",
        va="center",
        color=INK_2,
    )
    ax.set_yticks([2, 1, 0], ["safe", "leaky", "null"])
    ax.set_ylim(-0.6, 2.6)
    ax.grid(axis="y", visible=False)
    ax.set_title(title)
    lo = min(null_vals.min(), *(h[n][metric] for n in ("safe", "leaky")))
    hi = max(null_vals.max(), *(h[n][metric] for n in ("safe", "leaky")))
    pad = (hi - lo) * 0.25
    ax.set_xlim(lo - pad, hi + pad * 1.6)
fig.tight_layout()
save(fig, "07_homophily_dirichlet.png")
plt.show()

---
## Save the aggregate results

Every number above goes into one long-format CSV. `write_aggregate` checks the schema and refuses any patient-identifying column before anything touches disk.

In [ ]:
out = write_aggregate(rows, results_dir / "network_stats_task_a.csv")
saved = pd.read_csv(out)
print(f"wrote {len(saved)} rows -> {out.relative_to(_repo_root)}")
print(sorted(p.name for p in figures_dir.glob("07_*.png")))
saved.groupby("condition")["metric"].count()

## Findings

Numbers are from the run on Sep 25, 2026 (6,990 admissions, long-stay prevalence 0.253, primary graph configuration). "Null" means the mean of 10 rewired graphs, with the middle-95% range in brackets.

**1. The real graphs are "large worlds" made of tight clusters, and that structure is real.** All three graphs have identical degrees: mean 5.35, capped at 10 by mutual k-NN. The wiring is what differs:

| | safe | leaky | null |
|---|---|---|---|
| transitivity (closed triangles) | 0.498 | 0.660 | 0.001 [0.0008–0.0012] |
| average path in the largest piece (hops) | 9.6 | 11.9 | 5.3 [5.27–5.31] |
| largest piece (share of nodes) | 87.4% | 61.9% | 95.5% |

The rewired graph is a textbook small world with short paths and almost no triangles. The real graphs cluster about 500× more and have paths twice as long.

**2. The leaky graph has the "excess of closed triangles" PROJECT_GUIDE.md Module C flagged as a warning sign.** Its transitivity is 0.66 against 0.50 for safe. It is also more fragmented: 967 pieces against 408. This extends the mechanism from the notebook-05 follow-up. Seeing the future makes patients' vectors fuller and noisier, so fewer pairs choose each other mutually, and the pairs that do tend to sit in tight, closed cliques.

**3. Community structure is strong, stable, and tied to the outcome.** Modularity is 0.94 for safe and 0.98 for leaky, against 0.56 [0.553–0.560] for null. Louvain and Leiden largely agree: ARI 0.71 for safe and 0.82 for leaky, against 0.22 for null. Among Leiden groups of at least 20 patients, long-stay rates run from about 0% to 72% in the safe graph. Group membership explains **η² = 0.121** of the outcome variance for safe and 0.147 for leaky, against 0.007 [0.006–0.010] for null.

**4. Connected patients share outcomes far more than chance (issue #13).** Class-insensitive homophily is **0.209 (safe)** and 0.201 (leaky), against **0.011 [0.009–0.014]** for null. Put concretely, 40% of a long-stay patient's neighbors are also long-stay, against a 25% base rate. The CLMBR fingerprints tell the same story: Dirichlet energy is **0.683 (safe)** and 0.685 (leaky), against **0.953 [0.952–0.955]** for null. Graph neighbors already look alike to CLMBR.

**5. Centrality says nothing about the outcome.** Using any centrality measure as a stand-in predictor gives AUROC 0.47–0.51 across all graphs. The one exception is the leaky graph's eigenvector centrality at 0.55, and that measure is unreliable here anyway. Eigenvector centrality *localizes*: even inside the largest piece, over 97% of nodes score about 0 because the leading eigenvector concentrates on one dense cluster, which is expected in graphs this modular. Don't use it as a ranking of patients.

### What this means for the project

- **For long-stay prediction, leakage changed the *shape* of the graph but not how well it aligns with the outcome.** Safe and leaky have almost identical homophily (0.209 vs 0.201) and identical Dirichlet energy. If the leaky graph later *predicts* better (issues #15/#17, then Project 2), graph-level homophily won't explain it. The explanation would have to be local, for example in which specific edges changed. Only 52% of the safe graph's 18,698 edges (9,662) also appear in the leaky graph. That's worth noting now, before the prediction numbers exist, so it isn't read backwards later.
- **Low Dirichlet energy cuts both ways for Project 2.** Graph neighbors already have similar CLMBR fingerprints, so smoothing over them should reduce noise. It may also add little information CLMBR doesn't already have. This is the concrete version of the core question, and a reason to expect small gains rather than large ones.

### Caveats

- One graph configuration: diagnoses only, binary, TF-IDF, mutual k-NN with k = 10. The graph-variant sweep (Part 4, item 12) comes later.
- Homophily, group outcome rates, and centrality AUROC use *every* node's label. That's fine for describing the graph, but none of these numbers may be used to choose a model.
- The average path length is estimated from breadth-first search out of 500 sampled patients (9.559 vs the exact 9.558 on the safe graph). Everything else is exact.

### Done when
- [x] #11: degree distribution, structural statistics, and all four centralities (exact) for safe, leaky, and a 10-seed null ensemble
- [x] #12: Louvain + Leiden, agreement (ARI), modularity, and group outcome rates, each compared against the null range
- [x] #13: class-insensitive homophily (Lim et al. 2021) and normalized Dirichlet energy on centered CLMBR fingerprints
- [x] Only aggregate output written: `results/aggregate/network_stats_task_a.csv` (169 rows) and 5 figures